# CREST Benchmark Demo

**Columnar Rust Engine for Single-cell Transcriptomics**

This notebook demonstrates the full CREST pipeline on real scRNA-seq data,
benchmarks each operation, and compares against scanpy where applicable.

Dataset: **10x Genomics PBMC 10k v3** (11,769 cells, 33,538 genes)

---

## 0. Setup and Data Download

In [1]:
import polars as pl
import numpy as np
import time
import os
import subprocess
import crest

print(f"CREST version: {crest.__version__}")
print(f"Polars version: {pl.__version__}")
print(f"NumPy version: {np.__version__}")

CREST version: 0.1.0
Polars version: 1.39.3
NumPy version: 2.4.3


In [2]:
# Download PBMC 10k v3 from 10x Genomics (33 MB)
H5_URL = "https://cf.10xgenomics.com/samples/cell-exp/3.0.0/pbmc_10k_v3/pbmc_10k_v3_filtered_feature_bc_matrix.h5"
H5_PATH = "/tmp/pbmc_10k_v3.h5"
PARQUET_PATH = "/tmp/pbmc_10k_v3.parquet"

if not os.path.exists(H5_PATH):
    print("Downloading PBMC 10k v3 dataset...")
    subprocess.run(
        ["curl", "-sL", "-A", "Mozilla/5.0", "-o", H5_PATH, H5_URL],
        check=True,
    )
    print(f"Downloaded to {H5_PATH} ({os.path.getsize(H5_PATH) / 1e6:.1f} MB)")
else:
    print(f"Dataset already exists at {H5_PATH} ({os.path.getsize(H5_PATH) / 1e6:.1f} MB)")

Dataset already exists at /tmp/pbmc_10k_v3.h5 (37.5 MB)


## 1. I/O: HDF5 to Parquet Streaming Conversion

CREST converts 10x HDF5 to Parquet in fixed-size chunks.
Peak memory is independent of dataset size.

In [3]:
from crest.io import convert_h5_to_parquet_stream

if not os.path.exists(PARQUET_PATH):
    t0 = time.time()
    convert_h5_to_parquet_stream(H5_PATH, PARQUET_PATH, chunk_size=5000)
    io_time = time.time() - t0
    print(f"\nConversion time: {io_time:.2f}s")
else:
    print(f"Parquet already exists at {PARQUET_PATH}")

parquet_size = os.path.getsize(PARQUET_PATH) / 1e6
print(f"Parquet file size: {parquet_size:.1f} MB")

Parquet already exists at /tmp/pbmc_10k_v3.parquet
Parquet file size: 56.3 MB


In [4]:
# Load as LazyFrame (no materialization yet)
lf = pl.scan_parquet(PARQUET_PATH)

# Collect basic statistics
stats = lf.select(
    pl.col("cell_id").max().alias("max_cell"),
    pl.col("gene_id").max().alias("max_gene"),
    pl.len().alias("nnz"),
).collect()

n_cells = int(stats[0, "max_cell"]) + 1
n_genes = int(stats[0, "max_gene"]) + 1
nnz = int(stats[0, "nnz"])
sparsity = 1.0 - (nnz / (n_cells * n_genes))

print(f"Cells:    {n_cells:>10,}")
print(f"Genes:    {n_genes:>10,}")
print(f"Non-zero: {nnz:>10,}")
print(f"Sparsity: {sparsity:>10.1%}")
print(f"Dense equivalent: {n_cells * n_genes * 4 / 1e9:.2f} GB")
print(f"Sparse COO size:  {nnz * 12 / 1e6:.1f} MB")

Cells:        11,769
Genes:        33,536
Non-zero: 24,825,783
Sparsity:      93.7%
Dense equivalent: 1.58 GB
Sparse COO size:  297.9 MB


## 2. Quality Control

Compute per-cell QC metrics and filter low-quality cells.

In [5]:
# Collect into eager DataFrame for benchmarking
df = lf.collect()
print(f"Loaded {len(df):,} rows ({df.estimated_size('mb'):.1f} MB)")

Loaded 24,825,783 rows (284.1 MB)


In [6]:
# QC: total counts per cell
t0 = time.time()
df_qc = df.with_columns(
    pl.col("count").bio.qc_total_counts(pl.col("cell_id")).alias("total_counts"),
    pl.col("count").bio.qc_n_genes(pl.col("cell_id")).alias("n_genes_by_counts"),
)
qc_time = time.time() - t0
print(f"QC metrics computed in {qc_time:.4f}s")

# Per-cell summary (deduplicated)
cell_qc = df_qc.select("cell_id", "total_counts", "n_genes_by_counts").unique(subset=["cell_id"])
print(f"\nPer-cell QC summary ({len(cell_qc):,} cells):")
print(cell_qc.select(
    pl.col("total_counts").min().alias("min_counts"),
    pl.col("total_counts").median().alias("median_counts"),
    pl.col("total_counts").max().alias("max_counts"),
    pl.col("n_genes_by_counts").min().alias("min_genes"),
    pl.col("n_genes_by_counts").median().alias("median_genes"),
    pl.col("n_genes_by_counts").max().alias("max_genes"),
))

QC metrics computed in 0.0663s



Per-cell QC summary (11,769 cells):
shape: (1, 6)
┌────────────┬───────────────┬────────────┬───────────┬──────────────┬───────────┐
│ min_counts ┆ median_counts ┆ max_counts ┆ min_genes ┆ median_genes ┆ max_genes │
│ ---        ┆ ---           ┆ ---        ┆ ---       ┆ ---          ┆ ---       │
│ f32        ┆ f32           ┆ f32        ┆ u32       ┆ f64          ┆ u32       │
╞════════════╪═══════════════╪════════════╪═══════════╪══════════════╪═══════════╡
│ 501.0      ┆ 6517.0        ┆ 79534.0    ┆ 54        ┆ 1904.0       ┆ 7211      │
└────────────┴───────────────┴────────────┴───────────┴──────────────┴───────────┘


In [7]:
# Filter cells: min 200 genes, min 500 total counts
t0 = time.time()
df_filtered = df.with_columns(
    pl.col("count")
      .bio.filter_cells(pl.col("cell_id"), min_genes=200, min_counts=500.0)
      .alias("keep")
).filter(pl.col("keep")).drop("keep")
filter_time = time.time() - t0

cells_before = n_cells
cells_after = df_filtered["cell_id"].n_unique()
print(f"Filtering: {cells_before:,} -> {cells_after:,} cells ({filter_time:.4f}s)")
print(f"Removed {cells_before - cells_after:,} low-quality cells")
print(f"Rows: {len(df):,} -> {len(df_filtered):,}")

Filtering: 11,769 -> 11,537 cells (0.0486s)
Removed 232 low-quality cells
Rows: 24,825,783 -> 24,801,293


## 3. Normalization and Log Transform

CP10k normalization followed by ln(1+x) variance stabilization.

In [8]:
# Normalize + log1p
t0 = time.time()
df_norm = df_filtered.with_columns(
    pl.col("count")
      .bio.normalize_cpm(pl.col("cell_id"))
      .bio.log1p()
      .alias("logcpm")
)
norm_time = time.time() - t0
print(f"Normalize + log1p: {norm_time:.4f}s on {len(df_norm):,} rows")

# Spot check: first few values
print("\nSample output:")
print(df_norm.select("cell_id", "gene_id", "count", "logcpm").head(5))

Normalize + log1p: 0.0844s on 24,801,293 rows

Sample output:
shape: (5, 4)
┌─────────┬─────────┬───────┬──────────┐
│ cell_id ┆ gene_id ┆ count ┆ logcpm   │
│ ---     ┆ ---     ┆ ---   ┆ ---      │
│ u32     ┆ u32     ┆ f32   ┆ f32      │
╞═════════╪═════════╪═══════╪══════════╡
│ 0       ┆ 33508   ┆ 1.0   ┆ 1.71149  │
│ 0       ┆ 33505   ┆ 4.0   ┆ 2.952241 │
│ 0       ┆ 33503   ┆ 2.0   ┆ 2.309999 │
│ 0       ┆ 33502   ┆ 10.0  ┆ 3.836697 │
│ 0       ┆ 33501   ┆ 5.0   ┆ 3.164885 │
└─────────┴─────────┴───────┴──────────┘


## 4. Highly Variable Genes (HVG)

Select top 2000 genes by dispersion using Polars-native group-by.

In [9]:
from crest.core import BioFrame

# Wrap in BioFrame for HVG
obs = df_norm.select("cell_id").unique()
var = df_norm.select("gene_id").unique()
bf = BioFrame(X=df_norm.select("cell_id", "gene_id", pl.col("logcpm").alias("count")).lazy(), obs=obs, var=var)

from crest.pp import highly_variable_genes

t0 = time.time()
bf_hvg = highly_variable_genes(bf, n_top_genes=2000)
hvg_time = time.time() - t0

df_hvg = bf_hvg.X.collect()
n_genes_hvg = df_hvg["gene_id"].n_unique()
print(f"HVG selection: {n_genes:,} -> {n_genes_hvg:,} genes ({hvg_time:.4f}s)")
print(f"Rows after HVG filter: {len(df_hvg):,}")

HVG selection: 33,536 -> 2,000 genes (0.0819s)
Rows after HVG filter: 612,040


## 5. Scale (Zero-Center + Unit Variance)

Per-gene standardization accounting for structural zeros in sparse data.

In [10]:
n_obs = df_hvg["cell_id"].n_unique()

t0 = time.time()
df_scaled = df_hvg.with_columns(
    pl.col("count")
      .bio.scale(pl.col("gene_id"), n_obs=n_obs, max_value=10.0)
      .alias("scaled")
)
scale_time = time.time() - t0
print(f"Scale: {scale_time:.4f}s on {len(df_scaled):,} rows")

# Verify centering and clipping
print("\nScaled value range:")
print(df_scaled.select(
    pl.col("scaled").min().alias("min"),
    pl.col("scaled").mean().alias("mean"),
    pl.col("scaled").max().alias("max"),
    pl.col("scaled").std().alias("std"),
))

Scale: 0.0023s on 612,040 rows

Scaled value range:
shape: (1, 4)
┌──────────┬──────────┬──────┬──────────┐
│ min      ┆ mean     ┆ max  ┆ std      │
│ ---      ┆ ---      ┆ ---  ┆ ---      │
│ f32      ┆ f32      ┆ f32  ┆ f32      │
╞══════════╪══════════╪══════╪══════════╡
│ 0.042457 ┆ 7.907691 ┆ 10.0 ┆ 2.540969 │
└──────────┴──────────┴──────┴──────────┘


## 6. PCA (Sparse Randomized SVD)

Native Rust SVD operating directly on sparse COO triplets.
No dense matrix materialization.

In [11]:
# Remap cell_id and gene_id to dense contiguous indices for SVD
# (HVG filtering leaves sparse gene_ids like [5, 102, 5003, ...] which must be remapped to [0, 1, 2, ...])
unique_cells = df_hvg["cell_id"].unique().sort()
unique_genes = df_hvg["gene_id"].unique().sort()

cell_map = pl.DataFrame({"cell_id": unique_cells, "cell_idx": pl.Series(np.arange(len(unique_cells), dtype=np.uint32))})
gene_map = pl.DataFrame({"gene_id": unique_genes, "gene_idx": pl.Series(np.arange(len(unique_genes), dtype=np.uint32))})

df_remapped = (
    df_hvg
    .join(cell_map, on="cell_id", how="left")
    .join(gene_map, on="gene_id", how="left")
)

n_cells_svd = len(unique_cells)
n_genes_svd = len(unique_genes)

t0 = time.time()
agg = df_remapped.group_by(pl.lit(1).alias("batch")).agg([
    pl.col("cell_idx").alias("cell_id"),
    pl.col("gene_idx").alias("gene_id"),
    pl.col("count"),
])
agg_time = time.time() - t0
print(f"Aggregation + remap: {agg_time:.4f}s")

N_COMPS = 50
t0 = time.time()
agg_pca = agg.with_columns(
    pl.col("cell_id").bio.svd(
        pl.col("gene_id"), pl.col("count"),
        n_cells=n_cells_svd, n_genes=n_genes_svd, n_comps=N_COMPS
    ).alias("pca")
)
svd_time = time.time() - t0
print(f"SVD ({N_COMPS} components): {svd_time:.2f}s")

# Extract PCA into a multi-row DataFrame (one row per cell)
# SVD aggregate returns List(List(Float32)) in 1 row; unwrap to get individual cells
pca_inner = agg_pca["pca"][0]  # Series of List(Float32), one per cell
pca_df = pl.DataFrame({"pca": pca_inner})
print(f"PCA output: {len(pca_df)} cells x {len(pca_df['pca'][0])} components")

Aggregation + remap: 0.0013s


SVD (50 components): 0.83s
PCA output: 11537 cells x 50 components


## 7. Louvain Clustering + KNN

Build K-nearest neighbor graph and detect communities using Louvain algorithm.

In [12]:
K_NEIGHBORS = 15

# Neighbors, louvain, UMAP operate on the multi-row PCA DataFrame
# Each function internally builds a KD-tree or brute-force KNN from the PCA coords

t0 = time.time()
clust_result = pca_df.select(
    pl.col("pca").bio.louvain(n_neighbors=K_NEIGHBORS).alias("cluster")
)
nn_time = 0.0  # KNN built internally by louvain
louvain_time = time.time() - t0

clusters = clust_result["cluster"].to_numpy().astype(np.uint32)
n_clusters = len(np.unique(clusters))
print(f"Louvain clustering (includes internal KNN): {louvain_time:.2f}s")
print(f"Found {n_clusters} clusters")

# Cluster size distribution
unique, counts = np.unique(clusters, return_counts=True)
for c, n in sorted(zip(unique, counts), key=lambda x: -x[1])[:10]:
    print(f"  Cluster {c}: {n:,} cells ({n/len(clusters)*100:.1f}%)")
if n_clusters > 10:
    print(f"  ... and {n_clusters - 10} more clusters")

Louvain clustering (includes internal KNN): 3.55s
Found 76 clusters
  Cluster 39: 919 cells (8.0%)
  Cluster 27: 824 cells (7.1%)
  Cluster 37: 709 cells (6.1%)
  Cluster 26: 686 cells (5.9%)
  Cluster 46: 653 cells (5.7%)
  Cluster 25: 592 cells (5.1%)
  Cluster 58: 576 cells (5.0%)
  Cluster 55: 551 cells (4.8%)
  Cluster 34: 514 cells (4.5%)
  Cluster 47: 511 cells (4.4%)
  ... and 66 more clusters


## 8. UMAP

2D embedding using native Rust UMAP with Hogwild SGD.

In [13]:
# UMAP also takes the flat PCA DataFrame
t0 = time.time()
umap_result = pca_df.select(
    pl.col("pca").bio.umap(
        n_components=2,
        n_neighbors=K_NEIGHBORS,
        min_dist=0.1,
        n_epochs=200
    ).alias("umap")
)
umap_time = time.time() - t0
print(f"UMAP (200 epochs): {umap_time:.2f}s")

# UMAP returns aggregate List(List(Float32)); unwrap
umap_inner = umap_result["umap"][0]
umap_coords = np.array(umap_inner.to_list())
print(f"UMAP output: {umap_coords.shape}")

UMAP (200 epochs): 3.62s
UMAP output: (11537, 2)


## 9. UMAP

2D embedding using native Rust UMAP with Hogwild SGD.

In [14]:
t0 = time.time()
agg_umap = agg_pca.with_columns(
    pl.col("pca").bio.umap(
        n_components=2,
        n_neighbors=K_NEIGHBORS,
        min_dist=0.1,
        n_epochs=200
    ).alias("umap")
)
umap_time = time.time() - t0
print(f"UMAP (200 epochs): {umap_time:.2f}s")

umap_coords = np.array(agg_umap["umap"][0].to_list())
print(f"UMAP output: {umap_coords.shape}")

UMAP (200 epochs): 3.42s
UMAP output: (11537, 2)


In [15]:
# Visualization
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Colored by cluster
    ax = axes[0]
    scatter = ax.scatter(
        umap_coords[:, 0], umap_coords[:, 1],
        c=clusters, cmap="tab20", s=0.5, alpha=0.6, edgecolors="none"
    )
    ax.set_title(f"CREST UMAP — {n_cells_svd:,} cells, {n_clusters} clusters", fontsize=14)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal")

    # Colored by total counts (expression depth)
    ax = axes[1]
    # Compute per-cell total counts for coloring
    cell_totals = df_filtered.group_by("cell_id").agg(
        pl.col("count").sum().alias("total")
    ).sort("cell_id")["total"].to_numpy()
    # Trim to match SVD cell count
    cell_totals = cell_totals[:len(umap_coords)]
    scatter2 = ax.scatter(
        umap_coords[:, 0], umap_coords[:, 1],
        c=np.log1p(cell_totals), cmap="viridis", s=0.5, alpha=0.6, edgecolors="none"
    )
    ax.set_title("Colored by log(total counts)", fontsize=14)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal")
    plt.colorbar(scatter2, ax=ax, shrink=0.6, label="log1p(total_counts)")

    plt.tight_layout()
    plt.savefig("/tmp/crest_benchmark_umap.png", dpi=200, bbox_inches="tight")
    print("Saved plot to /tmp/crest_benchmark_umap.png")
    plt.show()

except ImportError:
    print("matplotlib not installed. Skipping visualization.")
    print(f"UMAP coordinate ranges: x=[{umap_coords[:,0].min():.2f}, {umap_coords[:,0].max():.2f}], "
          f"y=[{umap_coords[:,1].min():.2f}, {umap_coords[:,1].max():.2f}]")

Saved plot to /tmp/crest_benchmark_umap.png


## 10. Differential Expression

Welch t-test with Benjamini-Hochberg FDR correction.
Test: cluster 0 vs all other cells.

In [16]:
# Build group labels: assign cluster ID to each row (using remapped cell indices)
cluster_map = pl.DataFrame({
    "cell_id": pl.Series(np.arange(len(clusters), dtype=np.uint32)),
    "cluster": pl.Series(clusters),
})

# Join cluster labels onto the remapped expression data (flat, not aggregated)
df_de = df_remapped.select(
    pl.col("cell_idx").alias("cell_id"),
    pl.col("gene_idx").alias("gene_id"),
    pl.col("count"),
).join(cluster_map, on="cell_id", how="left")

# Find the largest cluster to use as target
target_cluster = int(unique[np.argmax(counts)])
print(f"Testing cluster {target_cluster} ({counts[np.argmax(counts)]:,} cells) vs rest")

# rank_genes_groups is an aggregate plugin — call via select (not group_by)
t0 = time.time()
de_result = df_de.select(
    pl.col("count").bio.rank_genes_groups(
        pl.col("cell_id"),
        pl.col("gene_id"),
        pl.col("cluster"),
        target_group=target_cluster
    ).alias("de")
)
de_time = time.time() - t0
print(f"Differential expression: {de_time:.2f}s")

# Extract and display top DE genes
de_genes = de_result["de"][0]
print(f"\nTop 15 differentially expressed genes (cluster {target_cluster} vs rest):")
print(f"{'Gene ID':>8}  {'t-stat':>8}  {'p-value':>10}  {'adj p':>10}  {'log2 FC':>8}")
print("-" * 52)
for i in range(min(15, len(de_genes))):
    row = de_genes[i].to_list()
    gene_id, t_stat, pval, adj_pval, log2fc = row
    print(f"{int(gene_id):>8}  {t_stat:>8.3f}  {pval:>10.2e}  {adj_pval:>10.2e}  {log2fc:>8.3f}")

Testing cluster 39 (919 cells) vs rest
Differential expression: 0.01s

Top 15 differentially expressed genes (cluster 39 vs rest):
 Gene ID    t-stat     p-value       adj p   log2 FC
----------------------------------------------------
     100    87.012    0.00e+00    0.00e+00     0.725
     105    78.821    0.00e+00    0.00e+00     0.848
     107    49.742    0.00e+00    0.00e+00     0.637
     246   -48.479    0.00e+00    0.00e+00    -1.989
     250   -42.576    0.00e+00    0.00e+00    -1.519
     549   -46.164    0.00e+00    0.00e+00    -1.821
    1080   -40.884    0.00e+00    0.00e+00    -1.643
    1168    55.199    0.00e+00    0.00e+00     0.533
    1209   -48.121    0.00e+00    0.00e+00    -1.792
    1265   102.319    0.00e+00    0.00e+00     0.895
    1412   -61.539    0.00e+00    0.00e+00    -2.089
    1513   -37.199    0.00e+00    0.00e+00    -1.491
    1683   -37.718    0.00e+00    0.00e+00    -1.332
    1695   -37.617    0.00e+00    0.00e+00    -1.200
    1875   -57.030   

## 11. Benchmark Summary

In [17]:
print("=" * 60)
print("  CREST Benchmark Summary")
print("  PBMC 10k v3 (10x Genomics)")
print("=" * 60)
print(f"  Dataset: {n_cells:,} cells, {n_genes:,} genes, {nnz:,} non-zeros")
print(f"  Sparsity: {sparsity:.1%}")
print()

benchmarks = [
    ("QC metrics (total counts + n_genes)", qc_time),
    ("Cell filtering (min_genes=200)", filter_time),
    ("Normalize CP10k + log1p", norm_time),
    ("HVG selection (top 2000)", hvg_time),
    ("Scale (zero-center + unit var)", scale_time),
    ("PCA (50 components, sparse SVD)", svd_time),
    ("Louvain clustering (KNN + communities)", louvain_time),
    ("UMAP (200 epochs)", umap_time),
    ("Rank genes groups (t-test + BH)", de_time),
]

total = 0
print(f"  {'Operation':<40} {'Time':>8}")
print(f"  {'-'*40} {'-'*8}")
for name, t in benchmarks:
    print(f"  {name:<40} {t:>7.2f}s")
    total += t
print(f"  {'-'*40} {'-'*8}")
print(f"  {'Total pipeline':>40} {total:>7.2f}s")
print()
print(f"  Clusters found: {n_clusters}")
print(f"  All operations ran in native Rust (no GIL).")
print("=" * 60)

  CREST Benchmark Summary
  PBMC 10k v3 (10x Genomics)
  Dataset: 11,769 cells, 33,536 genes, 24,825,783 non-zeros
  Sparsity: 93.7%

  Operation                                    Time
  ---------------------------------------- --------
  QC metrics (total counts + n_genes)         0.07s
  Cell filtering (min_genes=200)              0.05s
  Normalize CP10k + log1p                     0.08s
  HVG selection (top 2000)                    0.08s
  Scale (zero-center + unit var)              0.00s
  PCA (50 components, sparse SVD)             0.83s
  Louvain clustering (KNN + communities)      3.55s
  UMAP (200 epochs)                           3.42s
  Rank genes groups (t-test + BH)             0.01s
  ---------------------------------------- --------
                            Total pipeline    8.09s

  Clusters found: 76
  All operations ran in native Rust (no GIL).


## 12. Scale Test: 10M Synthetic Rows

Stress test preprocessing throughput on synthetic data.

In [18]:
N_SCALE = 10_000_000
np.random.seed(42)

big_df = pl.DataFrame({
    "cell_id": np.random.randint(0, 100_000, size=N_SCALE, dtype=np.uint32),
    "gene_id": np.random.randint(0, 30_000, size=N_SCALE, dtype=np.uint32),
    "count": np.random.exponential(scale=2.0, size=N_SCALE).astype(np.float32),
})
print(f"Synthetic dataset: {N_SCALE:,} rows, {big_df.estimated_size('mb'):.1f} MB")

scale_benchmarks = []

for name, expr in [
    ("log1p", lambda d: d.with_columns(pl.col("count").bio.log1p().alias("out"))),
    ("normalize_cpm", lambda d: d.with_columns(pl.col("count").bio.normalize_cpm(pl.col("cell_id")).alias("out"))),
    ("qc_total_counts", lambda d: d.with_columns(pl.col("count").bio.qc_total_counts(pl.col("cell_id")).alias("out"))),
    ("qc_n_genes", lambda d: d.with_columns(pl.col("count").bio.qc_n_genes(pl.col("cell_id")).alias("out"))),
    ("filter_cells", lambda d: d.with_columns(pl.col("count").bio.filter_cells(pl.col("cell_id"), min_genes=200).alias("out"))),
    ("scale", lambda d: d.with_columns(pl.col("count").bio.scale(pl.col("gene_id"), n_obs=100_000).alias("out"))),
]:
    t0 = time.time()
    _ = expr(big_df)
    elapsed = time.time() - t0
    throughput = N_SCALE / elapsed / 1e6
    scale_benchmarks.append((name, elapsed, throughput))

print(f"\n{'Function':<20} {'Time':>8} {'Throughput':>15}")
print(f"{'-'*20} {'-'*8} {'-'*15}")
for name, elapsed, throughput in scale_benchmarks:
    print(f"{name:<20} {elapsed:>7.4f}s {throughput:>12.0f}M rows/s")

Synthetic dataset: 10,000,000 rows, 114.4 MB



Function                 Time      Throughput
-------------------- -------- ---------------
log1p                 0.0986s          101M rows/s
normalize_cpm         0.1167s           86M rows/s
qc_total_counts       0.0930s          108M rows/s
qc_n_genes            0.0939s          106M rows/s
filter_cells          0.0910s          110M rows/s
scale                 0.1206s           83M rows/s


---

## Notes

**Data format**: CREST operates on sparse COO triplets (cell_id, gene_id, count) stored in Polars DataFrames. Only non-zero entries are stored. This is memory-efficient for scRNA-seq data, which is typically 90-95% sparse.

**Structural zeros vs informative zeros**: The COO format does not store zeros. In standard scRNA-seq analysis, zeros represent technical dropout and are accounted for via the `n_obs` parameter in `scale()` and through proper cell-total denominators in `normalize_cpm()`. For experiments where zero expression is biologically meaningful (e.g., gene knockouts), explicit zeros can be included as rows with `count=0.0` in the triplet format.

**Scaling considerations**: Preprocessing functions (log1p, normalize, QC, scale) scale linearly and work at any size. SVD and UMAP scale with the number of non-zero entries and cells respectively. The KD-tree based neighbors function scales well beyond 100k cells. The UMAP internal graph uses brute-force KNN, limiting it to approximately 50k cells.